<a href="https://colab.research.google.com/github/gregoirehendrix/Master_Thesis/blob/main/piping_losses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
CSP Field Piping Network — Layout Optimiser + Cascaded Thermal Loss Model
Author: Grégoire Hendrix
Date  : Apr 2026

Layout  : symmetric diamond grid, power block at centroid interstice
Thermal : NTU-based energy model, cascaded from leaves to PB
          with adiabatic mixing at every junction node
"""

import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


# =============================================================================
# 1.  LAYOUT OPTIMISER
# =============================================================================

class CSPOptimizerRows:
    def __init__(self, DIST, CAP):
        self.DIST = DIST
        self.CAP  = CAP

    def coord(self, p):
        i, j = p
        return (j * self.DIST, i * self.DIST)

    def manhattan(self, a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def generate_blue_towers(self, nb):
        n_side    = math.ceil(math.sqrt(nb))
        M         = n_side * 2 - 1
        positions = [(i, j) for i in range(0, M, 2)
                             for j in range(0, M, 2)]
        L  = (M - 1) * self.DIST
        cx, cy = L / 2, L / 2
        coords = np.array([self.coord(p) for p in positions])
        dist_c = np.sqrt((coords[:, 0] - cx)**2 + (coords[:, 1] - cy)**2)
        sel    = np.argsort(dist_c)[:nb]
        return [positions[i] for i in sel], M

    def group_by_rows(self, blue_pos):
        ordered = sorted(
            list(enumerate(blue_pos, start=1)),
            key=lambda x: (x[1][0], x[1][1])
        )
        clusters, row, buffer = [], None, []
        for idx, pos in ordered:
            r, c = pos
            if row is None:
                row = r
            if r != row:
                for k in range(0, len(buffer), self.CAP):
                    clusters.append(buffer[k:k + self.CAP])
                buffer, row = [], r
            buffer.append(idx)
        for k in range(0, len(buffer), self.CAP):
            clusters.append(buffer[k:k + self.CAP])
        return clusters

    def mst_with_hub(self, nodes, coords, hub):
        if len(nodes) == 1:
            return {}
        visited, unvisited, parent = {hub}, set(n for n in nodes if n != hub), {}
        while unvisited:
            best, best_d = None, math.inf
            for u in visited:
                for v in unvisited:
                    d = self.manhattan(coords[u], coords[v])
                    if d < best_d:
                        best_d, best = d, (u, v)
            u, v = best
            parent[v] = u
            visited.add(v)
            unvisited.remove(v)
        return parent

    def build_network_for_pb(self, blue_pos, clusters, pb):
        coords    = {0: self.coord(pb)}
        coords.update({i: self.coord(blue_pos[i - 1])
                       for i in range(1, len(blue_pos) + 1)})
        pb_coord  = coords[0]
        parent, cluster_map = {}, {}
        for cluster in clusters:
            hub = min(cluster, key=lambda i: self.manhattan(coords[i], pb_coord))
            for v, u in self.mst_with_hub(cluster, coords, hub).items():
                parent[v] = u
            parent[hub]      = 0
            cluster_map[hub] = cluster
        total = sum(self.manhattan(coords[b], coords[p]) for b, p in parent.items())
        return parent, coords, cluster_map, total

    def find_best_pb(self, blue_pos, M):
        clusters = self.group_by_rows(blue_pos)
        blue_set = set(blue_pos)
        candidates = [(i, j) for i in range(M) for j in range(M)
                      if (i, j) not in blue_set]
        best = None
        for pb in candidates:
            parent, coords, info, cost = self.build_network_for_pb(
                blue_pos, clusters, pb)
            if best is None or cost < best["cost"]:
                best = dict(pb=pb, parent=parent, coords=coords,
                            clusters=info, cost=cost)
        return best


# =============================================================================
# 2.  PIPE CHARGES & DIAMETERS
# =============================================================================

DIAM_TABLE = {
    1: '1 1/4"',
    2: '2"',
    3: '2 1/2"',
    4: '3"',
    5: '3"'
}

# Physical pipe properties by diameter class
DIAMETER_M = {
    '1 1/4"': 0.042,
    '2"'    : 0.060,
    '2 1/2"': 0.075,
    '3"'    : 0.089,
}

U_PIPE = {
    '1 1/4"': 1.2,
    '2"'    : 1.1,
    '2 1/2"': 1.0,
    '3"'    : 0.9,
}


def compute_pipe_charges_and_diameters(parent, nb_units, diameter_table):
    children = {i: [] for i in range(nb_units + 1)}
    for child, par in parent.items():
        children[par].append(child)

    charge = {}

    def dfs(node):
        if node != 0 and not children[node]:
            charge[node] = 1
            return 1
        total = 1 if node != 0 else 0
        for ch in children[node]:
            total += dfs(ch)
        charge[node] = total
        return total

    dfs(0)

    pipe_info = {}
    for b, p in parent.items():
        c    = charge[b]
        diam = diameter_table.get(c, diameter_table[max(diameter_table)])
        pipe_info[(b, p)] = dict(charge=c, diameter=diam)

    return pipe_info, charge, children


def compute_length_per_diameter(parent, coords, pipe_info):
    length_per_diam = {}
    for (child, par), info in pipe_info.items():
        diam = info["diameter"]
        x1, y1 = coords[child]
        x2, y2 = coords[par]
        dist   = abs(x1 - x2) + abs(y1 - y2)
        length_per_diam[diam] = length_per_diam.get(diam, 0) + dist
    return length_per_diam


# =============================================================================
# 3.  CASCADED NTU THERMAL MODEL
# =============================================================================

def ntu_outlet_temperature(T_in, T_amb, m_dot, Cp, U, D, L):
    """
    Single-pipe NTU model.
    T_out = T_amb + (T_in - T_amb) * exp(-NTU)
    NTU   = U * pi * D * L / (m_dot * Cp)
    """
    A   = math.pi * D * L
    NTU = U * A / (m_dot * Cp)
    return T_amb + (T_in - T_amb) * math.exp(-NTU)


def compute_cascaded_temperatures(
        parent, coords, pipe_info, children,
        T_hot, T_amb, m_dot_unit, Cp_salt,
        DIAMETER_M, U_PIPE):
    """
    Post-order DFS from leaves to PB.

    At each leaf   : T_in = T_hot (solar field outlet)
    At each node   : adiabatic mix of all incoming flows → T_in for next segment
    Returns
    -------
    node_T_out : dict  node_id → temperature leaving that node [K]
                 (for node 0 / PB, this is the mixed inlet temperature)
    pipe_results : dict  (child, parent) → {T_in, T_out, m_dot, L, NTU, Q_loss}
    T_PB : float  mixed temperature entering the power block [K]
    """
    node_T_out   = {}   # temperature of the fluid *after* the pipe from node → parent
    pipe_results = {}

    def dfs(node):
        if node != 0 and not children[node]:
            # ---- LEAF : fresh salt at T_hot ----
            node_T_out[node] = T_hot
            return

        # recurse first
        for ch in children[node]:
            dfs(ch)

        if node == 0:
            return   # PB handled separately below

        # ---- INTERNAL NODE : adiabatic mix of incoming children ----
        m_in_total = 0.0
        H_in_total = 0.0
        for ch in children[node]:
            T_ch   = node_T_out[ch]
            m_ch   = pipe_info[(ch, node)]["charge"] * m_dot_unit
            # apply NTU on the incoming segment ch → node
            diam   = pipe_info[(ch, node)]["diameter"]
            D_k    = DIAMETER_M[diam]
            U_k    = U_PIPE[diam]
            x1, y1 = coords[ch]
            x2, y2 = coords[node]
            L_k    = abs(x1 - x2) + abs(y1 - y2)
            T_out_seg = ntu_outlet_temperature(T_ch, T_amb, m_ch, Cp_salt, U_k, D_k, L_k)
            NTU_k     = U_k * math.pi * D_k * L_k / (m_ch * Cp_salt)
            Q_loss_k  = m_ch * Cp_salt * (T_ch - T_out_seg)

            pipe_results[(ch, node)] = dict(
                T_in   = T_ch,
                T_out  = T_out_seg,
                m_dot  = m_ch,
                L      = L_k,
                diam   = diam,
                NTU    = NTU_k,
                Q_loss = Q_loss_k,
            )

            m_in_total += m_ch
            H_in_total += m_ch * T_out_seg   # enthalpy proxy (Cp constant)

        # mixed temperature at this node
        T_mix = H_in_total / m_in_total
        node_T_out[node] = T_mix

    # start DFS from PB (node 0)
    for ch in children[0]:
        dfs(ch)

    # ---- FINAL SEGMENT : each hub → PB ----
    m_pb_total = 0.0
    H_pb_total = 0.0

    for ch in children[0]:
        T_ch   = node_T_out[ch]
        m_ch   = pipe_info[(ch, 0)]["charge"] * m_dot_unit
        diam   = pipe_info[(ch, 0)]["diameter"]
        D_k    = DIAMETER_M[diam]
        U_k    = U_PIPE[diam]
        x1, y1 = coords[ch]
        x2, y2 = coords[0]
        L_k    = abs(x1 - x2) + abs(y1 - y2)
        T_out_seg = ntu_outlet_temperature(T_ch, T_amb, m_ch, Cp_salt, U_k, D_k, L_k)
        NTU_k     = U_k * math.pi * D_k * L_k / (m_ch * Cp_salt)
        Q_loss_k  = m_ch * Cp_salt * (T_ch - T_out_seg)

        pipe_results[(ch, 0)] = dict(
            T_in   = T_ch,
            T_out  = T_out_seg,
            m_dot  = m_ch,
            L      = L_k,
            diam   = diam,
            NTU    = NTU_k,
            Q_loss = Q_loss_k,
        )

        m_pb_total += m_ch
        H_pb_total += m_ch * T_out_seg

    T_PB = H_pb_total / m_pb_total
    return node_T_out, pipe_results, T_PB


# =============================================================================
# 4.  REPORTING
# =============================================================================

def print_thermal_results(pipe_results, internal_to_visual, children, nb):
    """Print per-pipe thermal results, grouped by hub."""

    # build hub membership
    # hubs are direct children of node 0
    hubs = set(children[0])

    # member → hub mapping
    member_to_hub = {}
    for hub in hubs:
        for m in _all_descendants(hub, children):
            member_to_hub[m] = hub
        member_to_hub[hub] = hub

    # group pipe_results by hub
    hub_pipes = {h: [] for h in hubs}
    for (child, par), info in pipe_results.items():
        h = member_to_hub.get(child)
        if h is not None:
            hub_pipes[h].append((child, par, info))

    print("\n=== Per-hub cascaded thermal results ===")
    total_Q = 0.0

    hub_vis_list = sorted(hubs, key=lambda h: internal_to_visual[h])
    for hub in hub_vis_list:
        hub_vis = internal_to_visual[hub]
        print(f"\n--- Hub {hub_vis} ---")
        pipes = sorted(hub_pipes[hub],
                       key=lambda x: internal_to_visual.get(x[0], 0))
        hub_Q = 0.0
        for child, par, info in pipes:
            c_vis = internal_to_visual.get(child, "PB")
            p_vis = internal_to_visual.get(par,   "PB") if par != 0 else "PB"
            print(f"  [{c_vis} → {p_vis}]  "
                  f"diam={info['diam']:8s}  "
                  f"L={info['L']:6.1f} m  "
                  f"ṁ={info['m_dot']:6.2f} kg/s  "
                  f"T_in={info['T_in']-273.15:6.2f}°C  "
                  f"T_out={info['T_out']-273.15:6.2f}°C  "
                  f"Q_loss={info['Q_loss']/1e3:6.3f} kW")
            hub_Q   += info['Q_loss']
        total_Q += hub_Q
        print(f"  → Hub total loss : {hub_Q/1e3:.3f} kW")

    print(f"\n>>> TOTAL piping losses : {total_Q/1e6:.4f} MW")
    return total_Q


def _all_descendants(node, children):
    """Return all descendant node ids (including node itself)."""
    result = [node]
    for ch in children[node]:
        result.extend(_all_descendants(ch, children))
    return result


def compute_hub_pipe_details(parent, coords, pipe_info, clusters, nb_units):
    member_to_hub = {}
    for hub, members in clusters.items():
        for m in members:
            member_to_hub[m] = hub
    hub_pipes = {hub: {} for hub in clusters}
    for (child, par), info in pipe_info.items():
        diam = info["diameter"]
        x1, y1 = coords[child]
        x2, y2 = coords[par]
        dist   = abs(x1 - x2) + abs(y1 - y2)
        hub    = child if par == 0 else member_to_hub.get(child)
        if hub is not None and hub in hub_pipes:
            hub_pipes[hub][diam] = hub_pipes[hub].get(diam, 0) + dist
    return hub_pipes


def print_hub_pipe_details(hub_pipes, internal_to_visual):
    print("\n=== Pipe lengths per hub ===")
    result = [(internal_to_visual[h], d) for h, d in hub_pipes.items()]
    for hub_vis, diam_dict in sorted(result):
        print(f"\nHub {hub_vis} :")
        for diam, L in sorted(diam_dict.items()):
            print(f"  {diam} : {L:.2f} m")


# =============================================================================
# 5.  VISUALISATION
# =============================================================================

def make_visual_mapping(blue_pos):
    blue_for_display = sorted(
        [(i, pos) for i, pos in enumerate(blue_pos, start=1)],
        key=lambda x: (-x[1][0], x[1][1])
    )
    internal_to_visual = {
        internal: display
        for display, (internal, _) in enumerate(blue_for_display, start=1)
    }
    return blue_for_display, internal_to_visual


def plot_network(blue_pos, sol, DIST, M):
    parent  = sol["parent"]
    coords  = sol["coords"]
    clusters = sol["clusters"]
    hubs    = set(clusters.keys())

    fig, ax = plt.subplots(figsize=(13, 13))
    ax.set_aspect("equal")

    blue_for_display, internal_to_visual = make_visual_mapping(blue_pos)

    for display_id, (internal_id, (r, c)) in enumerate(blue_for_display, start=1):
        x, y = c * DIST, r * DIST
        edge  = "#FFD700" if internal_id in hubs else "black"
        lw    = 3         if internal_id in hubs else 0.7
        ax.add_patch(Rectangle((x, y), DIST, DIST,
                               facecolor="#0B3D91", edgecolor=edge, linewidth=lw))
        ax.text(x + DIST / 2, y + DIST / 2, str(display_id),
                ha="center", va="center", color="white")

    PB_SIZE = 0.3 * DIST
    xpb, ypb = coords[0]
    ax.add_patch(Rectangle(
        (xpb + (DIST - PB_SIZE) / 2, ypb + (DIST - PB_SIZE) / 2),
        PB_SIZE, PB_SIZE, facecolor="red", edgecolor="black", linewidth=2))

    for b, p in parent.items():
        bx, by = coords[b][0] + DIST / 2, coords[b][1] + DIST / 2
        px, py = coords[p][0] + DIST / 2, coords[p][1] + DIST / 2
        ax.plot([bx, px], [by, by], color="gray", linewidth=2)
        ax.plot([px, px], [by, py], color="gray", linewidth=2)

    ax.set_xlim(0, M * DIST)
    ax.set_ylim(0, M * DIST)
    ax.grid(True, alpha=0.3)
    plt.show()
    return internal_to_visual


def print_clusters_visual(clusters, mapping):
    print("\n=== Groupings ===")
    result = []
    for hub_int, members_int in clusters.items():
        hub_vis     = mapping[hub_int]
        members_vis = [mapping[m] for m in members_int]
        result.append((hub_vis, sorted(members_vis)))
    for hub, members in sorted(result):
        print(f"Hub {hub} → {members}")


# =============================================================================
# 6.  MAIN
# =============================================================================

if __name__ == "__main__":

    # --- Inputs ---
    nb     = int(input("Amount of CSP units per power block : "))
    CAP    = int(input("Maximal amount of units per group   : "))
    DIST_2 = 200          # centre-to-centre distance [m]
    DIST   = DIST_2 / 2

    # --- Thermal parameters ---
    T_hot     = 565 + 273.15    # K
    T_amb     =  25 + 273.15    # K
    T_C       = T_hot - 273.15
    Cp_salt   = 1443.0 + 0.172 * T_C   # J/kg/K
    m_dot_unit = 6.39            # kg/s per CSP unit

    # --- Layout optimisation ---
    opt      = CSPOptimizerRows(DIST, CAP)
    blue_pos, M = opt.generate_blue_towers(nb)
    sol      = opt.find_best_pb(blue_pos, M)

    # --- Pipe charges, diameters ---
    pipe_info, charge, children = compute_pipe_charges_and_diameters(
        sol["parent"], nb, DIAM_TABLE)

    # --- Layout metrics ---
    branch_lengths = [
        abs(sol["coords"][b][0] - sol["coords"][p][0]) +
        abs(sol["coords"][b][1] - sol["coords"][p][1])
        for b, p in sol["parent"].items()
    ]
    rms = math.sqrt(sum(L**2 for L in branch_lengths) / len(branch_lengths))

    # --- Plot ---
    mapping = plot_network(blue_pos, sol, DIST, M)

    # --- Layout summary ---
    print("Total piping length          :", sol["cost"], "m")
    print("Average piping length/unit   :", sol["cost"] / nb, "m")
    print("RMS piping length per branch :", rms, "m")

    print_clusters_visual(sol["clusters"], mapping)

    lengths = compute_length_per_diameter(sol["parent"], sol["coords"], pipe_info)
    print("\n=== Total length per diameter ===")
    for diam, L in lengths.items():
        print(f"  {diam} : {L:.2f} m")

    print_hub_pipe_details(
        compute_hub_pipe_details(sol["parent"], sol["coords"],
                                 pipe_info, sol["clusters"], nb),
        mapping)

    # --- Cascaded thermal model ---
    node_T_out, pipe_results, T_PB = compute_cascaded_temperatures(
        parent     = sol["parent"],
        coords     = sol["coords"],
        pipe_info  = pipe_info,
        children   = children,
        T_hot      = T_hot,
        T_amb      = T_amb,
        m_dot_unit = m_dot_unit,
        Cp_salt    = Cp_salt,
        DIAMETER_M = DIAMETER_M,
        U_PIPE     = U_PIPE,
    )

    total_Q = print_thermal_results(pipe_results, mapping, children, nb)

    print(f"\n>>> Salt temperature at PB inlet : {T_PB - 273.15:.3f} °C")
    print(f"    (drop vs solar field outlet  : {T_hot - T_PB:.3f} K)")
    print(f"    Total thermal losses         : {total_Q/1e6:.4f} MW")